# Question

**Peut-on prédir l'affluence horaire d'une ligne de tramway ?** 

# Data Exploration

*We describe our dataset and select the needed columns*

In [27]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

In [28]:
df_ref_tram_bus = pd.read_csv(r'C:\Users\chloe\Desktop\3A\dspap2026\Project\Data\pt_data\ref_tram_bus.csv',index_col = 0).rename(columns = {'IDT_PNT' : 'VAL_ARRET_CODE','COO_X_WGS84':'lon','COO_Y_WGS84':'lat'})
print(df_ref_tram_bus.info())
df_ref_tram_bus

<class 'pandas.DataFrame'>
Index: 9390 entries, 1 to 10212
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   VAL_ARRET_CODE   9390 non-null   int64  
 1   NOM_PNT          9390 non-null   str    
 2   NOM_HASTUS       397 non-null    str    
 3   COD_UTE          8596 non-null   str    
 4   FLG_PNT_ARR      9390 non-null   str    
 5   FLG_PNT_DPO      9390 non-null   str    
 6   FLG_PNT_GAR      9390 non-null   str    
 7   FLG_PNT_VIA      9390 non-null   str    
 8   IDT_LIE          8195 non-null   float64
 9   LIB_ADR          9177 non-null   str    
 10  COD_INSEE        8480 non-null   float64
 11  COD_PST          8480 non-null   float64
 12  LIB_CMN          8480 non-null   str    
 13  FLG_ADR_EN_FACE  9390 non-null   str    
 14  COO_X            9390 non-null   float64
 15  COO_Y            9390 non-null   float64
 16  lon              9390 non-null   float64
 17  lat              9390 non-nul

,VAL_ARRET_CODE,NOM_PNT,NOM_HASTUS,COD_UTE,FLG_PNT_ARR,FLG_PNT_DPO,FLG_PNT_GAR,FLG_PNT_VIA,IDT_LIE,LIB_ADR,COD_INSEE,COD_PST,LIB_CMN,FLG_ADR_EN_FACE,COO_X,COO_Y,lon,lat,CAP
1,30093,Gare de Vaise,NaN,UTV,O,N,N,N,10487.0,12 Rue Du 24 Mars 1852,69389.0,69009.0,LYON 9EME,N,791885.00,89700.00,4.804221,45.780508,33.0
2,30094,Gorge de Loup,NaN,NaN,N,N,N,N,NaN,56,NaN,NaN,NaN,N,792019.00,88141.00,4.805314,45.766451,178.0
5,30101,Perrache.,M_PERD,UTMA,O,N,N,N,30017.0,21 Pla Carnot,69382.0,69002.0,LYON 2EME,N,793756.28,86344.72,4.826904,45.749805,NaN
6,30103,Bellecour,NaN,UTMA,O,N,N,N,30019.0,1 Pla Le Viste,69382.0,69002.0,LYON 2EME,N,794204.33,87228.79,4.833019,45.757627,NaN
7,30106,Cordeliers-,NaN,UTMA,O,N,N,N,30020.0,24 Rue De La Republique,69382.0,69002.0,LYON 2EME,N,794395.00,87879.90,4.835735,45.763428,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10208,999,Garanjou,NaN,UTA,O,N,N,N,10482.0,63 Rue Ampere,69152.0,69310.0,PIERRE-BENITE,O,793088.78,81361.95,4.816311,45.705189,129.0
10209,214,Bois de l'Etoile,NaN,UTPE,O,N,N,N,10101.0,212 Rte De Sain Bel,69127.0,69280.0,MARCY L'ETOILE,O,785693.00,89524.00,4.724572,45.780645,102.0
10210,1795,Limonest Centre,NaN,UTV,O,N,N,N,10859.0,420 Ave General De Gaulle,69116.0,69760.0,LIMONEST,O,789150.52,95946.66,4.771565,45.837447,193.0
10211,2961,Terrasse,NaN,UTO,O,N,N,N,11381.0,7 Qua Jean Jacques Rousseau,69142.0,69350.0,LA MULATIERE,N,792886.00,86127.00,4.815637,45.748095,34.0


In [29]:
df_tramway = pd.read_csv(r'C:\Users\chloe\Desktop\3A\dspap2026\Project\Data\pt_data\tramway_indiv_15min\tramway_indiv_15min.csv')
df_tramway['VAL_DATE'] = pd.to_datetime(df_tramway['VAL_DATE']) 
df_tramway = df_tramway.merge(df_ref_tram_bus, how = 'inner', on = 'VAL_ARRET_CODE')

In [31]:
NOM_PNT_autriche = ['CDM Saint Priest']
NOM_PNT_norvege = ['Passage machine à laver UTTL', 'Via Part-Dieu Villette V1', 'CDM Meyzieu', 
                   'Retournement Charpennes', 'Retournement SPR Bel Air', 'Retournement Liberté',
                    'Garage Feyssine (voie 16)', 'Garage Feyssine (voie 24)',
                    'Passage machine à laver UTT', 'Retournement Perrache V1',
                    'Retournement Feyssine V2', 'Retournement Quai C. Bernard',
                    'Retournement Perrache V4'
                   ]

df_tramway = df_tramway[~df_tramway.NOM_PNT.isin(NOM_PNT_autriche + NOM_PNT_norvege)]
df_agg = df_tramway.groupby(['VAL_ARRET_CODE','LIG_NUMERO_SAE','CRS_SENS_TRAJET']).agg({'lon':'first','lat':'first','NOM_PNT':'first','COD_LIG_CLI':'first','Flow':'sum'})
df_agg = gpd.GeoDataFrame(df_agg, geometry=gpd.points_from_xy(df_agg.lon, df_agg.lat))
df_agg.crs = "EPSG:4326"
df_agg.explore('Flow',tiles = 'CartoDB positron',vmin = 0, vmax = df_agg.Flow.quantile(0.75))

c:\Users\chloe\Desktop\3A\dspap2026\Project\.venv\Lib\site-packages\geopandas\explore.py:383: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(scale_factor="{r}")


In [37]:
print(df_tramway.info())
print(df_tramway.columns)
df_tramway.head()

<class 'pandas.DataFrame'>
Index: 1915830 entries, 0 to 1916229
Data columns (total 25 columns):
 #   Column           Dtype         
---  ------           -----         
 0   Unnamed: 0       int64         
 1   LIG_NUMERO_SAE   int64         
 2   VAL_ARRET_CODE   int64         
 3   CRS_SENS_TRAJET  int64         
 4   COD_LIG_CLI      str           
 5   VAL_DATE         datetime64[us]
 6   Flow             int64         
 7   NOM_PNT          str           
 8   NOM_HASTUS       str           
 9   COD_UTE          str           
 10  FLG_PNT_ARR      str           
 11  FLG_PNT_DPO      str           
 12  FLG_PNT_GAR      str           
 13  FLG_PNT_VIA      str           
 14  IDT_LIE          float64       
 15  LIB_ADR          str           
 16  COD_INSEE        float64       
 17  COD_PST          float64       
 18  LIB_CMN          str           
 19  FLG_ADR_EN_FACE  str           
 20  COO_X            float64       
 21  COO_Y            float64       
 22  lon       

,Unnamed: 0,LIG_NUMERO_SAE,VAL_ARRET_CODE,CRS_SENS_TRAJET,COD_LIG_CLI,VAL_DATE,Flow,NOM_PNT,NOM_HASTUS,COD_UTE,...,LIB_ADR,COD_INSEE,COD_PST,LIB_CMN,FLG_ADR_EN_FACE,COO_X,COO_Y,lon,lat,CAP
0,0,520,32102,0,T1,2019-11-02 15:00:00,66,Perrache,NaN,UTT,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
1,1,520,32102,0,T1,2019-11-02 17:30:00,53,Perrache,NaN,UTT,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
2,2,520,32102,0,T1,2019-11-02 17:45:00,72,Perrache,NaN,UTT,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
3,3,520,32102,0,T1,2019-11-02 18:30:00,26,Perrache,NaN,UTT,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0
4,4,520,32102,0,T1,2019-11-03 04:45:00,1,Perrache,NaN,UTT,...,24 Crs De Verdun Perrache,69382.0,69002.0,LYON 2EME,N,793765.4,86322.08,4.827012,45.749599,204.0


In [42]:
df_tramway.isna().sum()

Unnamed: 0               0
LIG_NUMERO_SAE           0
VAL_ARRET_CODE           0
CRS_SENS_TRAJET          0
COD_LIG_CLI              0
VAL_DATE                 0
Flow                     0
NOM_PNT                  0
NOM_HASTUS         1845621
COD_UTE                  1
FLG_PNT_ARR              0
FLG_PNT_DPO              0
FLG_PNT_GAR              0
FLG_PNT_VIA              0
IDT_LIE                 38
LIB_ADR                  0
COD_INSEE                1
COD_PST                  1
LIB_CMN                  1
FLG_ADR_EN_FACE          0
COO_X                    0
COO_Y                    0
lon                      0
lat                      0
CAP                  16461
dtype: int64

In [44]:
cols_a_supprimer = [
    "Unnamed: 0", "FLG_PNT_GAR", "FLG_PNT_VIA",      # index parasite, colonnes constantes
    "LIB_ADR", "COD_PST", "LIB_CMN", "COD_INSEE",    # adresse / administratif
    "COO_X", "COO_Y",                                # redondant avec lon/lat
]
df_tramway = df_tramway.drop(columns=[c for c in cols_a_supprimer if c in df_tramway.columns])
df_tramway

,LIG_NUMERO_SAE,VAL_ARRET_CODE,CRS_SENS_TRAJET,COD_LIG_CLI,VAL_DATE,Flow,NOM_PNT,NOM_HASTUS,COD_UTE,FLG_PNT_ARR,FLG_PNT_DPO,IDT_LIE,FLG_ADR_EN_FACE,lon,lat,CAP
0,520,32102,0,T1,2019-11-02 15:00:00,66,Perrache,NaN,UTT,O,N,30537.0,N,4.827012,45.749599,204.0
1,520,32102,0,T1,2019-11-02 17:30:00,53,Perrache,NaN,UTT,O,N,30537.0,N,4.827012,45.749599,204.0
2,520,32102,0,T1,2019-11-02 17:45:00,72,Perrache,NaN,UTT,O,N,30537.0,N,4.827012,45.749599,204.0
3,520,32102,0,T1,2019-11-02 18:30:00,26,Perrache,NaN,UTT,O,N,30537.0,N,4.827012,45.749599,204.0
4,520,32102,0,T1,2019-11-03 04:45:00,1,Perrache,NaN,UTT,O,N,30537.0,N,4.827012,45.749599,204.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1916225,590,47378,1,T6,2020-03-30 20:15:00,1,Vinatier - T6,NaN,UTT,O,N,34533.0,N,4.894064,45.742515,202.0
1916226,590,47378,1,T6,2020-03-30 21:30:00,1,Vinatier - T6,NaN,UTT,O,N,34533.0,N,4.894064,45.742515,202.0
1916227,590,47378,1,T6,2020-03-30 22:00:00,1,Vinatier - T6,NaN,UTT,O,N,34533.0,N,4.894064,45.742515,202.0
1916228,903,3305,1,C3,2020-02-06 08:00:00,5,Gare Saint-Paul,NaN,UTS,O,N,11327.0,N,4.827918,45.766051,73.0


In [50]:
df_tramway["CAP"].describe()
df_tramway["CAP"].nunique()
df_tramway.groupby("LIG_NUMERO_SAE")["CAP"].agg(["min", "median", "max"])

,min,median,max
LIG_NUMERO_SAE,,,
520,4.0,204.0,358.0
530,20.0,223.0,353.0
540,0.0,182.0,332.0
550,0.0,180.0,358.0
560,10.0,22.0,293.0
570,20.0,165.0,345.0
580,101.0,132.0,163.0
590,10.0,202.0,295.0
903,73.0,81.5,90.0


Can't define 'CAP', what does it represent ?

**From now on we use df_tramway**

# Data Preparation

*We prepare our data to begin our work*